# Historical FPL simulation

Run cells in order. Milestone 1 is `select_initial_team()` then `resolve_current()`. Later cells call `advance()` for GW2+.

Fill `.env` first: Cohere, Supabase, and Databricks MLflow (`DATABRICKS_HOST`, `DATABRICKS_TOKEN`, `MLFLOW_TRACKING_URI=databricks`, `MLFLOW_EXPERIMENT_ID`). Apply `storage/migrations/001_init.sql` in Supabase (or `002_lock_on_lineup.sql` if `001` was already applied). An accepted `submit_lineup` locks the Gameweek.

In [1]:
from pathlib import Path
import sys

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")
print("repo", ROOT)

repo /Users/ojonyeagwu/Desktop/fpl_benchmark


In [2]:
from fpl_sim.tracking.mlflow_tracker import load_yaml

config = load_yaml("configs/baseline.yaml")
config

{'experiment': {'name': 'command_a_plus_baseline_v2'},
 'model': {'provider': 'cohere',
  'name': 'command-a-plus-05-2026',
  'temperature': 0.3,
  'seed': 42,
  'tool_choice': 'auto'},
 'skill': {'enabled': True,
  'version': 'baseline_v2',
  'path': 'skills/baseline/SKILL.md'},
 'tools': {'set': 'core_v1'},
 'memory': {'type': 'none'},
 'reflection': {'enabled': False},
 'subagents': {'enabled': False},
 'agent': {'max_tool_calls_per_gw': 40, 'max_turns_per_gw': 40},
 'world': {'dataset': 'final_fpl',
  'dataset_version': 'v1',
  'future_leakage_protection': True},
 'tracking': {'provider': 'databricks', 'experiment_name': '/Shared/fpl_sim'},
 'simulation': {'name': 'Command A+ Baseline',
  'season': '2025_26',
  'mode': 'historical',
  'rules_version': 'v1',
  'ruleset_path': 'configs/rules/fpl_2025_26.yaml'}}

In [3]:
from fpl_sim.world.context import WorldContext
from fpl_sim.world.historical_csv_world import HistoricalCSVWorld
from fpl_sim.world.repository import FPLWorldRepository

world = HistoricalCSVWorld()
repo = FPLWorldRepository(world, WorldContext(current_gw=1, phase="pre_deadline"))
assert repo.player_performance(1, last_n=5) == []
try:
    repo.gw_performance_map(1)
    raise SystemExit("leakage: GW1 points visible before deadline")
except PermissionError:
    print("pre-GW1 leakage check passed")
print("players in GW1 pool:", len(repo.current_players()))

pre-GW1 leakage check passed
players in GW1 pool: 637


In [4]:
import importlib

from dotenv import load_dotenv

import fpl_sim.storage.supabase as sb

load_dotenv(ROOT / ".env", override=True)
importlib.reload(sb)

store = sb.SupabaseStore()
print("supabase url", store.client.supabase_url)
print("supabase ok", store.ping())

supabase url https://<project>.supabase.co/
supabase ok True


In [5]:
n = store.wipe_all_simulations()
print(f"wiped {n} leftover simulation(s)")

# 002_lock_on_lineup.sql must be applied in the Supabase SQL editor once.
try:
    store.client.table("manager_gameweeks").select("mlflow_trace_id").limit(1).execute()
    schema_ok = True
except Exception:
    schema_ok = False
if not schema_ok:
    raise SystemExit(
        "Apply storage/migrations/002_lock_on_lineup.sql in the Supabase SQL editor, then re-run this cell."
    )
print("schema ok")

wiped 1 leftover simulation(s)
schema ok


In [6]:
from fpl_sim.harness.gameweek_runner import Simulation

sim = Simulation.from_config("configs/baseline.yaml", store=store, start_mlflow=True)
print("mlflow", sim.tracker.describe())
print("model", sim.config["model"]["name"])

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


mlflow uri=databricks experiment=/Users/<user>/fpl_sim id=<experiment-id> run=<run-id>
model command-a-plus-05-2026


## Milestone 1 — pick and lock GW1

Command A+ investigates with tools and locks GW1 with an accepted `submit_lineup` (after `submit_initial_squad`).

In [7]:
print(sim.select_initial_team())
print()
print(sim.squad_report())

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW1 DECISION

Transfers:
None

Chip:
None

Captain:
Evanilson
Vice:
Gvardiol

XI  Pickford         GK  Everton      £5.5m
GK2 Sels             GK  Nott'm Forest £5.0m
VC  Gvardiol         DEF Man City     £6.0m
XI  Milenković       DEF Nott'm Forest £5.5m
XI  Muñoz            DEF Crystal Palace £5.5m
XI  Saliba           DEF Arsenal      £6.0m
B1  Collins          DEF Brentford    £5.0m
XI  Caicedo          MID Chelsea      £5.5m
XI  Anderson         MID Nott'm Forest £5.5m
XI  Sarr             MID Crystal Palace £6.5m
XI  Mitoma           MID Brighton     £6.5m
B2  Tielemans        MID Aston Villa  £6.0m
C   Evanilson        FWD Bournemouth  £7.0m
XI  Welbeck          FWD Brighton     £6.5m
B3  Delap            FWD Chelsea      £6.5m


In [14]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import format_squad_pitch_html, gw_opponent_labels

gw_row = sim.store.get_gameweek(sim.simulation_id, sim.current_gw)
squad = sim.store.list_squad(gw_row["id"])
display(
    HTML(
        format_squad_pitch_html(
            squad,
            sim.repo.current_player_map(),
            bank=gw_row.get("bank"),
            team_value=gw_row.get("team_value"),
            gw=gw_row.get("gw"),
            opponents_by_team=gw_opponent_labels(sim.repo, int(gw_row["gw"])),
        )
    )
)

## Milestone 1b — resolve GW1

In [ ]:
print(sim.resolve_current())

In [15]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import counted_points_from_store, format_squad_pitch_html, gw_opponent_labels

gw_row = sim.store.get_gameweek(sim.simulation_id, sim.current_gw)
squad = sim.store.list_squad(gw_row["id"])
points = counted_points_from_store(sim.store, gw_row["id"])
display(
    HTML(
        format_squad_pitch_html(
            squad,
            sim.repo.current_player_map(),
            bank=gw_row.get("bank"),
            team_value=gw_row.get("team_value"),
            gw=gw_row.get("gw"),
            opponents_by_team=gw_opponent_labels(sim.repo, int(gw_row["gw"])),
            points_by_player=points,
            title=f"GW{gw_row['gw']} result",
            gross_points=gw_row.get("gross_points"),
            transfer_hit=gw_row.get("transfer_hit"),
            net_points=gw_row.get("net_points"),
        )
    )
)

## Sequential Gameweeks

From GW2 onward, run the next two cells in order. Re-run the same pair through GW38. `lock_next()` asks the model to lock the upcoming Gameweek; `resolve_current()` scores it.

In [16]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
display(HTML(simulation_squad_pitch_html(sim)))

GW2 DECISION

Transfers:
Sels -> Ramsdale
Delap -> Wood

Transfer hit: -4
Chip:
None

Captain:
Evanilson
Vice:
Wood


In [17]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
display(HTML(simulation_result_pitch_html(sim)))

GW2 RESULT

Gross points:       55
Transfer hit:       -4
Net points:         51
FPL average:        51
Highest:            140
Season total:       88

Captain: Evanilson
Captain contribution: 4

Bench points: 10


## GW3

In [20]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW3 DECISION

Transfers:
Evanilson -> Raúl

Chip:
None

Captain:
Wood
Vice:
Caicedo


In [21]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
display(HTML(simulation_result_pitch_html(sim)))

GW3 RESULT

Gross points:       35
Transfer hit:       0
Net points:         35
FPL average:        48
Highest:            118
Season total:       123

Captain: Wood
Captain contribution: 4

Bench points: 7


## GW 4

In [22]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW4 DECISION

Transfers:
Gvardiol -> Burn

Chip:
None

Captain:
Burn
Vice:
Muñoz


In [23]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
display(HTML(simulation_result_pitch_html(sim)))

GW4 RESULT

Gross points:       54
Transfer hit:       0
Net points:         54
FPL average:        63
Highest:            139
Season total:       177

Captain: Burn
Captain contribution: 12

Bench points: 1


## GW 5

In [24]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW5 DECISION

Transfers:
None

Chip:
None

Captain:
Wood
Vice:
Caicedo


In [25]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW5 RESULT

Gross points:       30
Transfer hit:       0
Net points:         30
FPL average:        42
Highest:            112
Season total:       207

Captain: Wood
Captain contribution: 4

Bench points: 2


## GW 6

In [27]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW6 DECISION

Transfers:
Tielemans -> Beto
Welbeck -> Iwobi

Chip:
None

Captain:
Wood
Vice:
Caicedo


In [28]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW6 RESULT

Gross points:       27
Transfer hit:       0
Net points:         27
FPL average:        46
Highest:            124
Season total:       234

Captain: Wood
Captain contribution: 4

Bench points: 12


## GW 7

In [29]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW7 DECISION

Transfers:
Collins -> Guéhi

Chip:
None

Captain:
Burn
Vice:
Caicedo


In [30]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW7 RESULT

Gross points:       66
Transfer hit:       0
Net points:         66
FPL average:        60
Highest:            135
Season total:       300

Captain: Burn
Captain contribution: 22

Bench points: 2


## GW 8

In [31]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW8 DECISION

Transfers:
Wood -> Welbeck

Chip:
None

Captain:
Guéhi
Vice:
Sarr


In [32]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW8 RESULT

Gross points:       55
Transfer hit:       0
Net points:         55
FPL average:        56
Highest:            138
Season total:       355

Captain: Guéhi
Captain contribution: 8

Bench points: 2


## GW 9

In [33]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW9 DECISION

Transfers:
None

Chip:
None

Captain:
Caicedo
Vice:
Saliba


In [34]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW9 RESULT

Gross points:       27
Transfer hit:       0
Net points:         27
FPL average:        46
Highest:            124
Season total:       382

Captain: Caicedo
Captain contribution: 4

Bench points: 6


## GW 10

In [35]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW10 DECISION

Transfers:
Raúl -> Strand Larsen

Chip:
None

Captain:
Caicedo
Vice:
Welbeck


In [36]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW10 RESULT

Gross points:       60
Transfer hit:       0
Net points:         60
FPL average:        65
Highest:            135
Season total:       442

Captain: Caicedo
Captain contribution: 20

Bench points: 5


## GW 11

In [38]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW11 DECISION

Transfers:
None

Chip:
None

Captain:
Caicedo
Vice:
Welbeck


In [39]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW11 RESULT

Gross points:       40
Transfer hit:       0
Net points:         40
FPL average:        38
Highest:            120
Season total:       482

Captain: Caicedo
Captain contribution: 4

Bench points: 1


## GW 12

In [42]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

sim.agent.max_tool_calls = 60
sim.agent.max_turns = 60
print(sim.agent.max_tool_calls, sim.agent.max_turns)

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

60 60
Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW12 DECISION

Transfers:
Ramsdale -> Sels

Chip:
None

Captain:
Caicedo
Vice:
Muñoz


In [43]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW12 RESULT

Gross points:       76
Transfer hit:       0
Net points:         76
FPL average:        39
Highest:            134
Season total:       558

Captain: Muñoz
Captain contribution: 28

Bench points: 10


## GW 13

In [44]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

sim.agent.max_tool_calls = 60
sim.agent.max_turns = 60
print(sim.agent.max_tool_calls, sim.agent.max_turns)

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

60 60
GW13 DECISION

Transfers:
None

Chip:
None

Captain:
Welbeck
Vice:
Pickford


In [45]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW13 RESULT

Gross points:       20
Transfer hit:       0
Net points:         20
FPL average:        35
Highest:            123
Season total:       578

Captain: Welbeck
Captain contribution: 4

Bench points: 4


## GW 14

In [46]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.agent.max_tool_calls, sim.agent.max_turns)

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

60 60
Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW14 DECISION

Transfers:
Caicedo -> Tonali

Chip:
None

Captain:
Welbeck
Vice:
Anderson


In [47]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW14 RESULT

Gross points:       62
Transfer hit:       0
Net points:         62
FPL average:        58
Highest:            138
Season total:       640

Captain: Welbeck
Captain contribution: 2

Bench points: 6


## GW 15

In [48]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

sim.agent.max_tool_calls = 60
sim.agent.max_turns = 60
print(sim.agent.max_tool_calls, sim.agent.max_turns)

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

60 60
Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW15 DECISION

Transfers:
Strand Larsen -> Isidor
Sarr -> Baleba

Chip:
None

Captain:
Welbeck
Vice:
Isidor


In [49]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW15 RESULT

Gross points:       35
Transfer hit:       0
Net points:         35
FPL average:        49
Highest:            133
Season total:       675

Captain: Welbeck
Captain contribution: 4

Bench points: 2


## GW 16

In [50]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW16 DECISION

Transfers:
Muñoz -> Tarkowski
Beto -> Raúl
Sels -> Sánchez

Chip:
None

Captain:
Raúl
Vice:
Guéhi


In [51]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW16 RESULT

Gross points:       17
Transfer hit:       0
Net points:         17
FPL average:        60
Highest:            149
Season total:       692

Captain: Raúl
Captain contribution: 4

Bench points: 17


## GW 17

In [52]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

sim.agent.max_tool_calls = 60
sim.agent.max_turns = 60
print(sim.agent.max_tool_calls, sim.agent.max_turns)

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

60 60
GW17 DECISION

Transfers:
Baleba -> Wilson

Chip:
None

Captain:
Guéhi
Vice:
Raúl


In [53]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW17 RESULT

Gross points:       33
Transfer hit:       0
Net points:         33
FPL average:        66
Highest:            148
Season total:       725

Captain: Guéhi
Captain contribution: 2

Bench points: 2


## GW 18

In [54]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

sim.agent.max_tool_calls = 60
sim.agent.max_turns = 60
print(sim.agent.max_tool_calls, sim.agent.max_turns)

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

60 60
GW18 DECISION

Transfers:
Iwobi -> Caicedo

Chip:
None

Captain:
Guéhi
Vice:
Tarkowski


In [55]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW18 RESULT

Gross points:       41
Transfer hit:       0
Net points:         41
FPL average:        44
Highest:            118
Season total:       766

Captain: Guéhi
Captain contribution: 4

Bench points: 8


## GW 19

In [56]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW19 DECISION

Transfers:
Burn -> Mykolenko

Chip:
None

Captain:
Wilson
Vice:
Guéhi


In [57]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW19 RESULT

Gross points:       47
Transfer hit:       0
Net points:         47
FPL average:        40
Highest:            112
Season total:       813

Captain: Wilson
Captain contribution: 4

Bench points: 5


## GW 20

In [58]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

sim.agent.max_tool_calls = 60
sim.agent.max_turns = 60
print(sim.agent.max_tool_calls, sim.agent.max_turns)

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

60 60
GW20 DECISION

Transfers:
Caicedo -> Rogers

Chip:
None

Captain:
Welbeck
Vice:
Tonali


In [59]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW20 RESULT

Gross points:       29
Transfer hit:       0
Net points:         29
FPL average:        42
Highest:            117
Season total:       842

Captain: Welbeck
Captain contribution: 2

Bench points: 8


## GW 21

In [60]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW21 DECISION

Transfers:
None

Chip:
None

Captain:
Pickford
Vice:
Guéhi


In [61]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW21 RESULT

Gross points:       46
Transfer hit:       0
Net points:         46
FPL average:        48
Highest:            118
Season total:       888

Captain: Pickford
Captain contribution: 6

Bench points: 17


## GW 22

In [62]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW22 DECISION

Transfers:
Mitoma -> Caicedo
Milenković -> Collins

Chip:
None

Captain:
Collins
Vice:
Rogers


In [63]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW22 RESULT

Gross points:       52
Transfer hit:       0
Net points:         52
FPL average:        40
Highest:            111
Season total:       940

Captain: Collins
Captain contribution: 2

Bench points: 12


## GW 23

In [64]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW23 DECISION

Transfers:
Welbeck -> Strand Larsen

Chip:
None

Captain:
Saliba
Vice:
Caicedo


In [65]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW23 RESULT

Gross points:       38
Transfer hit:       0
Net points:         38
FPL average:        44
Highest:            115
Season total:       978

Captain: Saliba
Captain contribution: 2

Bench points: 16


## GW 24

In [66]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW24 DECISION

Transfers:
None

Chip:
None

Captain:
Isidor
Vice:
Sánchez


In [67]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW24 RESULT

Gross points:       30
Transfer hit:       0
Net points:         30
FPL average:        55
Highest:            126
Season total:       1008

Captain: Isidor
Captain contribution: 2

Bench points: 11


## GW 25

In [68]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW25 DECISION

Transfers:
Mykolenko -> Milenković
Isidor -> Welbeck

Chip:
None

Captain:
Raúl
Vice:
Saliba


In [69]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW25 RESULT

Gross points:       47
Transfer hit:       0
Net points:         47
FPL average:        58
Highest:            147
Season total:       1055

Captain: Raúl
Captain contribution: 10

Bench points: 5


## GW 26

In [70]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW26 DECISION

Transfers:
Collins -> Lacroix

Chip:
None

Captain:
Rogers
Vice:
Welbeck


In [71]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW26 RESULT

Gross points:       39
Transfer hit:       0
Net points:         39
FPL average:        58
Highest:            138
Season total:       1094

Captain: Rogers
Captain contribution: 4

Bench points: 10


## GW 27

In [72]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW27 DECISION

Transfers:
None

Chip:
None

Captain:
Saliba
Vice:
Anderson


In [73]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW27 RESULT

Gross points:       50
Transfer hit:       0
Net points:         50
FPL average:        45
Highest:            117
Season total:       1144

Captain: Saliba
Captain contribution: 4

Bench points: 5


## GW 28

In [75]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW28 DECISION

Transfers:
None

Chip:
None

Captain:
Raúl
Vice:
Rogers


In [76]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW28 RESULT

Gross points:       54
Transfer hit:       0
Net points:         54
FPL average:        53
Highest:            125
Season total:       1198

Captain: Raúl
Captain contribution: 4

Bench points: 12


## GW 29

In [77]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW29 DECISION

Transfers:
Lacroix -> Mitchell
Tonali -> Ndiaye

Chip:
None

Captain:
Guéhi
Vice:
Rogers


In [78]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW29 RESULT

Gross points:       51
Transfer hit:       0
Net points:         51
FPL average:        54
Highest:            143
Season total:       1249

Captain: Guéhi
Captain contribution: 2

Bench points: 2


## GW 30

In [79]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW30 DECISION

Transfers:
Milenković -> Collins
Strand Larsen -> Zirkzee

Chip:
None

Captain:
Guéhi
Vice:
Saliba


In [80]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW30 RESULT

Gross points:       34
Transfer hit:       0
Net points:         34
FPL average:        43
Highest:            110
Season total:       1283

Captain: Guéhi
Captain contribution: 4

Bench points: 6


## GW 31

### Had to use different API key cuz current one ran out of free requests

In [84]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display
import os
from dotenv import load_dotenv
from cohere import ClientV2

load_dotenv(ROOT / ".env", override=True)
sim.agent.client = ClientV2(api_key=os.environ["COHERE_API_KEY"])

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW31 DECISION

Transfers:
None

Chip:
None

Captain:
Wilson
Vice:
Raúl


In [85]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW31 RESULT

Gross points:       77
Transfer hit:       0
Net points:         77
FPL average:        38
Highest:            109
Season total:       1360

Captain: Wilson
Captain contribution: 18

Bench points: 2


## GW 32

In [86]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW32 DECISION

Transfers:
Ndiaye -> Van Hecke
Guéhi -> M.Fernandes

Chip:
None

Captain:
Welbeck
Vice:
Wilson


In [87]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW32 RESULT

Gross points:       31
Transfer hit:       0
Net points:         31
FPL average:        46
Highest:            132
Season total:       1391

Captain: Welbeck
Captain contribution: 4

Bench points: 8


## GW 33

In [88]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW33 DECISION

Transfers:
None

Chip:
None

Captain:
Welbeck
Vice:
Caicedo


In [89]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW33 RESULT

Gross points:       60
Transfer hit:       0
Net points:         60
FPL average:        66
Highest:            177
Season total:       1451

Captain: Welbeck
Captain contribution: 16

Bench points: 15


## GW 34

In [90]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
Cohere per-minute limit (429). Waiting 30s then retrying chat (2/8).
GW34 DECISION

Transfers:
Zirkzee -> Evanilson

Chip:
None

Captain:
Evanilson
Vice:
Welbeck


In [91]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW34 RESULT

Gross points:       30
Transfer hit:       0
Net points:         30
FPL average:        36
Highest:            114
Season total:       1481

Captain: None
Captain contribution: 0

Bench points: 0


## GW 35

In [93]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW35 DECISION

Transfers:
Welbeck -> João Pedro

Chip:
None

Captain:
João Pedro
Vice:
Saliba


In [94]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW35 RESULT

Gross points:       45
Transfer hit:       0
Net points:         45
FPL average:        50
Highest:            129
Season total:       1526

Captain: João Pedro
Captain contribution: 16

Bench points: 6


## GW 36

In [95]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW36 DECISION

Transfers:
None

Chip:
None

Captain:
Raúl
Vice:
Evanilson


In [96]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW36 RESULT

Gross points:       56
Transfer hit:       0
Net points:         56
FPL average:        65
Highest:            145
Season total:       1582

Captain: Evanilson
Captain contribution: 2

Bench points: 3


## GW 37

In [97]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

Cohere per-minute limit (429). Waiting 20s then retrying chat (1/8).
GW37 DECISION

Transfers:
Evanilson -> Beto

Chip:
None

Captain:
João Pedro
Vice:
Raúl


In [98]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
#display(HTML(simulation_result_pitch_html(sim)))

GW37 RESULT

Gross points:       46
Transfer hit:       0
Net points:         46
FPL average:        53
Highest:            131
Season total:       1628

Captain: Raúl
Captain contribution: 2

Bench points: 9


## GW 38

In [99]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_squad_pitch_html

print(sim.lock_next())
#display(HTML(simulation_squad_pitch_html(sim)))

GW38 DECISION

Transfers:
Collins -> Mukiele
Rogers -> E.Le Fée

Chip:
None

Captain:
Beto
Vice:
Raúl


In [100]:
display(HTML(simulation_squad_pitch_html(sim)))

In [101]:
import importlib
import fpl_sim.harness.reports as reports
from IPython.display import HTML, display

importlib.reload(reports)
from fpl_sim.harness.reports import simulation_result_pitch_html

print(sim.resolve_current())
display(HTML(simulation_result_pitch_html(sim)))

🏃 View run command_a_plus_baseline_v2 at: https://<workspace>.cloud.databricks.com/ml/experiments/<experiment-id>/runs/<run-id>
🧪 View experiment at: https://<workspace>.cloud.databricks.com/ml/experiments/<experiment-id>
GW38 RESULT

Gross points:       31
Transfer hit:       0
Net points:         31
FPL average:        41
Highest:            117
Season total:       1659

Captain: Beto
Captain contribution: 2

Bench points: 8
